In [ ]:
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from point_e.diffusion.configs import DIFFUSION_CONFIGS, diffusion_from_config
from point_e.diffusion.sampler import PointCloudSampler
from point_e.models.download import load_checkpoint
from point_e.models.configs import MODEL_CONFIGS, model_from_config
from point_e.util.plotting import plot_point_cloud
from point_e.util.cae_preprocessor import CAEPreprocessor
from point_e.util.cae_visualizer import visualize_point_cloud, visualize_point_cloud_simple, print_point_cloud_stats

In [ ]:
# Load and preprocess the CAE format file
print("Loading CAE file...")
cae_file = "data/sample_cae.txt"

preprocessor = CAEPreprocessor()
preprocessor.parse_file(cae_file)

print(f"Loaded {len(preprocessor.grids)} points")
print(f"Loaded {len(preprocessor.elements)} elements")
print(f"Found {len(preprocessor.pshells)} unique PIDs")

In [ ]:
# Convert to PointCloud format
pc_base = preprocessor.to_point_cloud()

# Print statistics
print_point_cloud_stats(pc_base)

In [ ]:
# Visualize the base point cloud
print("Visualizing base point cloud...")
fig = visualize_point_cloud_simple(
    pc_base,
    title="Base Point Cloud from CAE Format",
    figsize=(10, 10),
    point_size=50
)
plt.show()

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Note: The following code requires downloading model checkpoints
# For this example to work with upsampling, you would need to:
# 1. Have models trained on similar data
# 2. Ensure the point cloud has enough points for upsampling

print("Note: Upsampling requires pre-trained models.")
print("For this example, we'll skip the actual upsampling step.")
print("To use upsampling with CAE data:")
print("  1. Ensure your CAE file has enough points (1024+ recommended)")
print("  2. Load the upsampler model as shown in text2pointcloud.ipynb")
print("  3. Pass the base point cloud through the upsampler")


In [ ]:
# Example structure for upsampling (requires models to be available):
# 
# # Create upsampler model
# upsampler_model = model_from_config(MODEL_CONFIGS['upsample'], device)
# upsampler_model.eval()
# upsampler_diffusion = diffusion_from_config(DIFFUSION_CONFIGS['upsample'])
# 
# # Load checkpoint
# upsampler_model.load_state_dict(load_checkpoint('upsample', device))
# 
# # Create sampler
# sampler = PointCloudSampler(
#     device=device,
#     models=[upsampler_model],
#     diffusions=[upsampler_diffusion],
#     num_points=[4096 - 1024],  # Upsample from 1024 to 4096
#     aux_channels=['R', 'G', 'B'],
#     guidance_scale=[0.0],
# )
# 
# # Perform upsampling
# # Note: This requires the base point cloud to have the right format

print("Example structure shown above in comments.")

In [ ]:
# Visualize from multiple angles
print("Creating multi-view visualization...")
fig = visualize_point_cloud(
    pc_base,
    title="CAE Point Cloud - Multiple Views",
    grid_size=2,
    figsize=(12, 12),
    point_size=50
)
plt.show()

In [ ]:
# Save the point cloud
output_file = "cae_pointcloud.npz"
pc_base.save(output_file)
print(f"Point cloud saved to {output_file}")

# Also save as PLY format for use in other tools
with open("cae_pointcloud.ply", 'wb') as f:
    pc_base.write_ply(f)
print("Point cloud saved to cae_pointcloud.ply")